# 02 - Data Preprocessing

Preprocess the CSIC 2010 HTTP dataset for deep learning WAF experiments with PyTorch.

In [ ]:
# Import core data handling libraries.
import os
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Make notebook output easier to read.
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
# Define project paths used by this preprocessing pipeline.
RAW_DATA_PATH = "../data/raw/csic_2010.csv"
PROCESSED_DATA_PATH = "../data/processed/csic_processed.csv"
TRAIN_SPLIT_PATH = "../data/splits/train.csv"
VAL_SPLIT_PATH = "../data/splits/val.csv"
TEST_SPLIT_PATH = "../data/splits/test.csv"
SCALER_PATH = "../models/scaler.pkl"

# Create output directories if they do not already exist.
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/splits", exist_ok=True)
os.makedirs("../models", exist_ok=True)

In [ ]:
# Load the CSIC 2010 HTTP dataset.
df = pd.read_csv(RAW_DATA_PATH)

# Drop unnamed index columns that may appear after CSV export.
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False, regex=True)]

# Fix the common misspelling 'lenght' if present in the source dataset.
if "lenght" in df.columns and "length" not in df.columns:
    df = df.rename(columns={"lenght": "length"})

# Show basic dataset information before cleaning.
print("Initial dataset shape:", df.shape)
display(df.head())
display(df.info())

In [ ]:
# Store the original shape so cleaning impact is visible.
shape_before_cleaning = df.shape

# Drop exact duplicate HTTP request rows.
df = df.drop_duplicates().copy()

# Fill missing text fields with empty strings for safe regex/string processing.
df["content"] = df["content"].fillna("").astype(str)
df["URL"] = df["URL"].fillna("").astype(str)

# Fill missing request length values with 0 and ensure the column is numeric.
df["length"] = pd.to_numeric(df["length"], errors="coerce").fillna(0)

# Ensure the target column is numeric and remove rows without a valid target.
df["classification"] = pd.to_numeric(df["classification"], errors="coerce")
df = df.dropna(subset=["classification"]).copy()
df["classification"] = df["classification"].astype(int)

# Print the shape before and after cleaning.
print("Shape before cleaning:", shape_before_cleaning)
print("Shape after cleaning:", df.shape)

In [ ]:
# Build helper text by combining URL and HTTP body content.
combined_text = df["URL"] + " " + df["content"]

# Create URL and content length features.
df["url_length"] = df["URL"].str.len()
df["content_length"] = df["content"].str.len()

# Detect common SQL injection indicators using case-insensitive regex.
sql_pattern = r"\b(SELECT|UNION|INSERT|DROP)\b|--|;"
df["has_sql"] = combined_text.str.contains(sql_pattern, flags=re.IGNORECASE, regex=True).astype(int)

# Detect common XSS indicators using case-insensitive regex.
xss_pattern = r"<script|javascript:|onerror|alert\s*\(\s*\)"
df["has_xss"] = combined_text.str.contains(xss_pattern, flags=re.IGNORECASE, regex=True).astype(int)

# Detect path traversal indicators in the URL only.
path_traversal_pattern = r"\.\./|\.\.\\|%2e%2e"
df["has_path_traversal"] = df["URL"].str.contains(path_traversal_pattern, flags=re.IGNORECASE, regex=True).astype(int)

# Label-encode HTTP methods such as GET and POST for model input.
method_encoder = LabelEncoder()
df["Method"] = df["Method"].fillna("UNKNOWN").astype(str)
df["method_encoded"] = method_encoder.fit_transform(df["Method"])

# Count special characters in the URL: ' " < > ; = & % +
special_char_pattern = r"['\"<>;=&%+]"
df["special_char_count"] = df["URL"].str.count(special_char_pattern)

# Display engineered features for a quick sanity check.
display(df[["URL", "content", "url_length", "content_length", "has_sql", "has_xss", "has_path_traversal", "method_encoded", "special_char_count", "length", "classification"]].head())

In [ ]:
# Select final model input features and target label.
features = [
    "url_length",
    "content_length",
    "has_sql",
    "has_xss",
    "has_path_traversal",
    "method_encoded",
    "special_char_count",
    "length",
]
target = "classification"

# Keep only the features and target needed by PyTorch model notebooks.
processed_df = df[features + [target]].copy()

# Verify that all selected model columns are numeric.
display(processed_df.head())
display(processed_df.dtypes)

In [ ]:
# Split feature matrix and target vector.
X = processed_df[features]
y = processed_df[target]

# First split: 70% train and 30% temporary data, stratified by class.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

# Second split: divide the temporary data equally into 15% validation and 15% test.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

# Print split sizes.
print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

In [ ]:
# Normalize numerical features using MinMaxScaler.
# Important: fit only on training data to avoid validation/test data leakage.
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=features, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=features, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features, index=X_test.index)

# Save the fitted scaler so inference code can apply identical preprocessing.
joblib.dump(scaler, SCALER_PATH)
print(f"Scaler saved to: {SCALER_PATH}")

In [ ]:
# Recombine scaled features with labels for each split.
train_df = X_train_scaled.copy()
train_df[target] = y_train.values

val_df = X_val_scaled.copy()
val_df[target] = y_val.values

test_df = X_test_scaled.copy()
test_df[target] = y_test.values

# Create a full processed dataset using the train-fitted scaler.
processed_scaled_df = pd.DataFrame(scaler.transform(processed_df[features]), columns=features, index=processed_df.index)
processed_scaled_df[target] = processed_df[target].values

# Save train, validation, and test splits to CSV files.
train_df.to_csv(TRAIN_SPLIT_PATH, index=False)
val_df.to_csv(VAL_SPLIT_PATH, index=False)
test_df.to_csv(TEST_SPLIT_PATH, index=False)

# Save the full processed dataset to CSV.
processed_scaled_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Train split saved to: {TRAIN_SPLIT_PATH}")
print(f"Validation split saved to: {VAL_SPLIT_PATH}")
print(f"Test split saved to: {TEST_SPLIT_PATH}")
print(f"Full processed dataset saved to: {PROCESSED_DATA_PATH}")

In [ ]:
# Print final preprocessing summary.
print("Final preprocessing summary")
print("=" * 32)
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))
print("\nFeature list:")
for feature in features:
    print(f"- {feature}")

print("\nClass distribution - Train:")
print(train_df[target].value_counts(normalize=False).sort_index())
print(train_df[target].value_counts(normalize=True).sort_index())

print("\nClass distribution - Validation:")
print(val_df[target].value_counts(normalize=False).sort_index())
print(val_df[target].value_counts(normalize=True).sort_index())

print("\nClass distribution - Test:")
print(test_df[target].value_counts(normalize=False).sort_index())
print(test_df[target].value_counts(normalize=True).sort_index())